In [2]:
import tensorflow as tf
print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"Success! Found {len(gpus)} GPU(s): {gpus}")
else:
    print("!!! ERROR: TensorFlow did not find a GPU in this environment.")

TypeError: Unable to convert function return value to a Python type! The signature was
	() -> handle

In [1]:
import pandas as pd
import numpy as np
import re
import os

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks, utils

# ----------------------------
# Enable Mixed Precision & XLA (for speed)
# ----------------------------
tf.keras.mixed_precision.set_global_policy('mixed_float16')
tf.config.optimizer.set_jit(True)

# ----------------------------
# Configuration
# ----------------------------
MAX_LEN = 120
EMBEDDING_DIM = 128
BATCH_SIZE = 1024      # Maximize GPU throughput
EPOCHS_PHASE1 = 20      # Balanced training
EPOCHS_PHASE2 = 30      # Full-data fine-tuning
N_FOLDS = 5
RANDOM_STATE = 42
TARGET_PER_CLASS = 20000   # For balancing

# Emotion labels - REMOVED 'error'
EMOTION_LABELS = ['neutral', 'sadness', 'fear', 'happiness', 'anger', 'disgust', 'surprise']
LABEL_TO_IDX = {label: idx for idx, label in enumerate(EMOTION_LABELS)}
IDX_TO_LABEL = {idx: label for label, idx in LABEL_TO_IDX.items()}


# ----------------------------
# Preprocessing (Bulgarian-safe)
# ----------------------------
def preprocess_bulgarian_text(text):
    if pd.isna(text) or not isinstance(text, str):
        return ""
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'[^\u0400-\u04FFa-zA-Z\s.,!?;:]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()


def load_dataset(file_path):
    df = pd.read_csv(file_path)
    df = df.dropna(subset=['text', 'emotion'])
    df['emotion'] = df['emotion'].astype(str).str.lower().str.strip()
    df = df[df['emotion'].isin(EMOTION_LABELS)]  # 'error' excluded
    df['clean_text'] = df['text'].apply(preprocess_bulgarian_text)
    df = df[df['clean_text'].str.len() > 0]
    return df


# ----------------------------
# Tokenization
# ----------------------------
def train_bpe_tokenizer(texts, vocab_size=35000):
    tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(vocab_size=vocab_size, special_tokens=["[PAD]", "[UNK]"])
    tokenizer.train_from_iterator(texts, trainer=trainer)
    return tokenizer


def encode_texts(tokenizer, text_list):
    encodings = tokenizer.encode_batch(text_list)
    pad_id = tokenizer.token_to_id("[PAD]")
    input_ids = [
        (enc.ids[:MAX_LEN] + [pad_id] * MAX_LEN)[:MAX_LEN]
        for enc in encodings
    ]
    return np.array(input_ids, dtype=np.int32)


# ----------------------------
# Balanced Dataset Creation (Undersampling Only)
# ----------------------------
def create_balanced_dataset(X, y, target_per_class=20000):
    X_bal_list = []
    y_bal_list = []

    for label_idx in range(len(EMOTION_LABELS)):
        mask = (y == label_idx)
        X_class = X[mask]
        y_class = y[mask]
        n_class = len(y_class)

        if n_class == 0:
            continue

        if n_class > target_per_class:
            # Undersample majority (e.g., 'neutral')
            indices = np.random.choice(n_class, size=target_per_class, replace=False)
            X_bal_list.append(X_class[indices])
            y_bal_list.append(y_class[indices])
        else:
            # Keep all minority classes
            X_bal_list.append(X_class)
            y_bal_list.append(y_class)

    if not X_bal_list:
        raise ValueError("No data to balance!")

    X_bal = np.concatenate(X_bal_list, axis=0)
    y_bal = np.concatenate(y_bal_list, axis=0)

    # Shuffle
    shuffle_idx = np.random.permutation(len(X_bal))
    return X_bal[shuffle_idx], y_bal[shuffle_idx]


# ----------------------------
# tf.data Pipeline
# ----------------------------
def create_tf_dataset(X, y, batch_size, is_training=False):
    dataset = tf.data.Dataset.from_tensor_slices((X, y))
    if is_training:
        dataset = dataset.shuffle(buffer_size=100_000)
    dataset = dataset.batch(batch_size, drop_remainder=False)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


# ----------------------------
# Model
# ----------------------------
def build_model(vocab_size, num_classes):
    inputs = layers.Input(shape=(MAX_LEN,), dtype='int32')
    x = layers.Embedding(vocab_size, EMBEDDING_DIM, mask_zero=True)(inputs)

    x = layers.Bidirectional(layers.LSTM(384, return_sequences=True, dropout=0.3))(x)
    x = layers.LayerNormalization()(x)
    x = layers.Bidirectional(layers.LSTM(384, return_sequences=True, dropout=0.3))(x)
    x = layers.LayerNormalization()(x)

    attn = layers.MultiHeadAttention(num_heads=8, key_dim=64)(x, x)
    x = layers.Add()([x, attn])
    x = layers.LayerNormalization()(x)

    max_pool = layers.GlobalMaxPooling1D()(x)
    avg_pool = layers.GlobalAveragePooling1D()(x)
    x = layers.Concatenate()([max_pool, avg_pool])

    x = layers.Dense(512, activation='swish', kernel_initializer='he_normal')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='swish', kernel_initializer='he_normal')(x)
    x = layers.Dropout(0.4)(x)

    outputs = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)
    return models.Model(inputs=inputs, outputs=outputs)


# ----------------------------
# Focal Loss
# ----------------------------
def focal_loss(gamma=2.0, alpha=0.75):
    def loss(y_true, y_pred):
        # IMPROVEMENT #3: Ensure y_true is float32 to match y_pred
        y_true = tf.cast(y_true, dtype=tf.float32)

        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)

        ce = -y_true * tf.math.log(y_pred)
        weight = alpha * y_true * tf.pow(1 - y_pred, gamma)

        return tf.reduce_mean(tf.reduce_sum(weight * ce, axis=1))
    return loss


# ----------------------------
# Main Training Loop
# ----------------------------
def main():
    print("🚀 High-Performance Bulgarian Emotion Classifier (Balanced Training)")
    print(f"Classes: {EMOTION_LABELS}")
    print(f"Batch Size: {BATCH_SIZE} | Mixed Precision: ON | XLA: ON")

    df_main = load_dataset('FINAL.csv')
    print(f"✅ Loaded {len(df_main):,} samples.")
    print("Distribution:\n", df_main['emotion'].value_counts())

    texts = df_main['clean_text'].tolist()
    labels = df_main['emotion'].map(LABEL_TO_IDX).values

    print("🔤 Training BPE tokenizer...")
    tokenizer = train_bpe_tokenizer(texts, vocab_size=35000)
    tokenizer.save("bg_emotion_tokenizer_balanced.json")
    vocab_size = tokenizer.get_vocab_size()

    X = encode_texts(tokenizer, texts)
    y = labels

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n{'='*20} Fold {fold+1}/{N_FOLDS} {'='*20}")

        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # IMPROVEMENT #2: Calculate class weights on the fold's training data only
        class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
        class_weight_dict = dict(enumerate(class_weights))
        print("Fold-specific Class Weights:", {IDX_TO_LABEL[i]: f"{w:.2f}" for i, w in class_weight_dict.items()})


        y_train_cat = utils.to_categorical(y_train, len(EMOTION_LABELS))
        y_val_cat = utils.to_categorical(y_val, len(EMOTION_LABELS))

        val_ds = create_tf_dataset(X_val, y_val_cat, BATCH_SIZE)

        # === PHASE 1: Balanced Training ===
        print("⚖️  Phase 1: Training on balanced dataset...")
        X_bal, y_bal = create_balanced_dataset(X_train, y_train, TARGET_PER_CLASS)
        y_bal_cat = utils.to_categorical(y_bal, len(EMOTION_LABELS))
        train_ds_bal = create_tf_dataset(X_bal, y_bal_cat, BATCH_SIZE, is_training=True)

        tf.keras.backend.clear_session()
        model = build_model(vocab_size, len(EMOTION_LABELS))
        model.compile(
            optimizer=optimizers.Adam(learning_rate=1e-3, clipnorm=1.0),
            loss=focal_loss(gamma=2.0, alpha=0.75),
            metrics=['accuracy']
        )

        model.fit(
            train_ds_bal,
            validation_data=val_ds,
            epochs=EPOCHS_PHASE1,
            callbacks=[
                callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
                callbacks.TerminateOnNaN()
            ],
            verbose=1
        )

        # === PHASE 2: Fine-tune on Full Data ===
        print("🔍 Phase 2: Fine-tuning on full dataset...")
        train_ds_full = create_tf_dataset(X_train, y_train_cat, BATCH_SIZE, is_training=True)

        model.compile(
            optimizer=optimizers.Adam(learning_rate=5e-4, clipnorm=1.0),  # Lower LR
            loss=focal_loss(gamma=2.0, alpha=0.75),
            metrics=['accuracy']
        )

        model.fit(
            train_ds_full,
            validation_data=val_ds,
            epochs=EPOCHS_PHASE2,
            callbacks=[
                callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
                callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
                callbacks.TerminateOnNaN()
            ],
            class_weight=class_weight_dict,
            verbose=1
        )

        # === Evaluation ===
        y_pred = model.predict(val_ds, verbose=0)
        y_pred_classes = np.argmax(y_pred, axis=1)
        report = classification_report(
            y_val, y_pred_classes,
            target_names=EMOTION_LABELS,
            output_dict=True,
            zero_division=0
        )
        macro_f1 = report['macro avg']['f1-score']
        print(f"✅ Fold {fold+1} Macro F1: {macro_f1:.4f}")

        model.save(f"bg_emotion_balanced_fold_{fold+1}.keras")

    print("\n🎉 Training complete. Models saved in .keras format.")


if __name__ == "__main__":
    main()

ImportError: DLL load failed while importing _multiarray_umath: The specified module could not be found.

ImportError: DLL load failed while importing _multiarray_umath: The specified module could not be found.

ImportError: numpy.core._multiarray_umath failed to import

ImportError: numpy.core.umath failed to import

TypeError: Unable to convert function return value to a Python type! The signature was
	() -> handle

In [ ]:
# ----------------------------
# ENVIRONMENT & LIBRARY SETUP
# ----------------------------
import numpy as np, pandas as pd, random, tensorflow as tf, sys
import re
import os

# Imports for BPE Tokenization
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# Standard ML/DL Imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from tensorflow.keras import layers, Model, utils, optimizers
import matplotlib.pyplot as plt

# Enable Mixed Precision & XLA (for speed)
tf.keras.mixed_precision.set_global_policy('mixed_float16')
tf.config.optimizer.set_jit(True)

# ----------------------------
# Configuration
# ----------------------------
seed = 42
train_pkl = "FINAL_but_cooler.pkl"
test_csv  = "NLP_features_engineered_but_even_cooler.csv"
text_col  = "text"
label_col = "emotion"
test_text_col  = "Sentence"
test_label_col = "Read emotions"

# --- Model/Data Constants ---
MAX_LEN = 120
EMBEDDING_DIM = 128
VOCAB_SIZE = 35000
BATCH_SIZE = 128
TOKENIZER_PATH = 'bpe_tokenizer.json'
MODEL_PATH = 'best_bpe_focal.keras'

# Set seeds for reproducibility
np.random.seed(seed); random.seed(seed); tf.random.set_seed(seed)
EVAL_MODE = '--evaluate' in sys.argv

print(f"--- Starting Script (Evaluation Mode: {EVAL_MODE}) ---")

# ----------------------------
# 1. BULGARIAN PREPROCESSING
# ----------------------------

def preprocess_bulgarian_text(text):
    """Removes URLs/noise and keeps Cyrillic/Latin characters."""
    if pd.isna(text) or not isinstance(text, str):
        return ""
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # Keep Cyrillic, Latin, spaces, and essential punctuation
    text = re.sub(r'[^\u0400-\u04FFa-zA-Z\s.,!?;:]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()


# ----------------------------
# 2. BPE TOKENIZATION FUNCTIONS
# ----------------------------

def train_bpe_tokenizer(texts, vocab_size=VOCAB_SIZE):
    """Trains a BPE tokenizer and saves it to file."""
    tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(vocab_size=vocab_size, special_tokens=["[PAD]", "[UNK]"])
    tokenizer.train_from_iterator(texts, trainer=trainer)
    return tokenizer

def encode_texts(tokenizer, text_list):
    """Encodes texts to padded integer sequences using BPE tokenizer."""
    encodings = tokenizer.encode_batch(text_list)

    # Get the ID for the [PAD] token
    pad_id = tokenizer.token_to_id("[PAD]")

    input_ids = np.array([enc.ids for enc in encodings], dtype=np.int32)

    # Pad sequences to MAX_LEN
    input_ids = tf.keras.preprocessing.sequence.pad_sequences(
        input_ids,
        maxlen=MAX_LEN,
        padding='post',
        truncating='post',
        value=pad_id
    )
    return input_ids

# ----------------------------
# 3. CUSTOM FOCAL LOSS FUNCTION
# ----------------------------
def focal_loss(gamma=2.0, alpha=0.75):
    """Focal Loss for classification, critical for extreme imbalance."""
    def loss(y_true, y_pred):
        # y_true must be one-hot (categorical)
        y_true = tf.cast(y_true, dtype=tf.float32)

        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)

        # Cross-entropy part
        ce = -y_true * tf.math.log(y_pred)
        # Weighting factor: alpha * (1 - p)^gamma
        weight = alpha * tf.pow(1 - y_pred, gamma) * y_true

        return tf.reduce_mean(tf.reduce_sum(weight * ce, axis=1))
    return loss


# ----------------------------
# 4. PLOTTING UTILITY FUNCTIONS
# ----------------------------

def plot_confusion_matrix(y_true, y_pred, class_names, filename='test_confusion_matrix_bpe.png'):
    """Calculates, plots, and saves the confusion matrix as a PNG file."""
    cm = confusion_matrix(y_true, y_pred)
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    plt.figure(figsize=(10, 8))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title("Confusion Matrix")
    plt.colorbar()

    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45)
    plt.yticks(tick_marks, class_names)

    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, f"{cm[i, j]}\n({cm_normalized[i, j]:.2f})",
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.savefig(filename)
    plt.close()
    print(f"\n[INFO] Saved Confusion Matrix to {filename}")


def plot_training_history(history, filename='training_history_bpe.png'):
    """Plots and saves the loss curves over training epochs (Train vs Val Loss)."""
    train_loss = history.history['loss']
    val_loss = history.history['val_loss']

    plt.figure(figsize=(10, 6))
    plt.plot(train_loss, label='Training Loss', color='blue')
    plt.plot(val_loss, label='Validation/Test Loss', color='orange')
    plt.title('Training vs Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.savefig(filename)
    plt.close()
    print(f"\n[INFO] Saved Training History Plot to {filename}")

# ----------------------------
# 5. MODEL ARCHITECTURE
# ----------------------------
def build_model(vocab_size):
    """Defines the stacked Bi-LSTM + Multi-Head Attention architecture."""
    inputs = layers.Input(shape=(MAX_LEN,), dtype='int32')

    # 1. Fully Trainable Embedding Layer (BPE structure)
    x = layers.Embedding(vocab_size, EMBEDDING_DIM, mask_zero=True)(inputs)

    # 2. Stacked Bi-LSTMs (Wider, Increased Dropout)
    x = layers.Bidirectional(layers.LSTM(384, return_sequences=True, dropout=0.3))(x)
    x = layers.LayerNormalization()(x) # LayerNorm stabilizes stacked LSTMs
    x = layers.Bidirectional(layers.LSTM(384, return_sequences=True, dropout=0.3))(x)
    x = layers.LayerNormalization()(x)

    # 3. Multi-Head Attention
    attn = layers.MultiHeadAttention(num_heads=8, key_dim=64)(x, x)
    x = layers.Add()([x, attn])
    x = layers.LayerNormalization()(x)

    # 4. Aggregation (Sentence Embedding)
    max_pool = layers.GlobalMaxPooling1D()(x)
    avg_pool = layers.GlobalAveragePooling1D()(x)
    x = layers.Concatenate()([max_pool, avg_pool])

    # 5. Classifier Head
    x = layers.Dense(512, activation='swish', kernel_initializer='he_normal')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='swish', kernel_initializer='he_normal')(x)
    x = layers.Dropout(0.4)(x)

    # Final output layer (dtype='float32' for stability with mixed precision)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)
    return Model(inputs=inputs, outputs=outputs)


# ----------------------------
# MAIN EXECUTION
# ----------------------------
# Load data
df_train = pd.read_csv(train_pkl)[[text_col, label_col]].dropna(subset=[text_col, label_col])
df_test  = pd.read_csv(test_csv)[[test_text_col, test_label_col]].dropna(subset=[test_text_col, test_label_col])
df_test  = df_test.rename(columns={test_text_col: text_col, test_label_col: label_col})

# Apply preprocessing to text columns
df_train[text_col] = df_train[text_col].apply(preprocess_bulgarian_text)
df_test[text_col] = df_test[text_col].apply(preprocess_bulgarian_text)

# Filter out empty texts after cleaning
df_train = df_train[df_train[text_col].str.len() > 0]
df_test = df_test[df_test[text_col].str.len() > 0]


print("Train size:", len(df_train), "Test size:", len(df_test))

# Encode labels
encoder = LabelEncoder()
y_all = encoder.fit_transform(df_train[label_col])
num_classes = len(encoder.classes_)
EMOTION_LABELS = encoder.classes_

# Split data
X_train, X_val, y_train, y_val = train_test_split(
    df_train[text_col], y_all, test_size=0.2, random_state=seed, stratify=y_all
)
# Encode test labels
y_test = encoder.transform(df_test[label_col])

# --- Tokenizer Setup ---
if os.path.exists(TOKENIZER_PATH):
    print(f"Loading BPE tokenizer from {TOKENIZER_PATH}...")
    tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
else:
    print("Training BPE tokenizer (This may take a minute)...")
    tokenizer = train_bpe_tokenizer(X_train.tolist(), vocab_size=VOCAB_SIZE)
    tokenizer.save(TOKENIZER_PATH)
    print(f"Tokenizer saved to {TOKENIZER_PATH}.")

# Encode and Pad sequences using BPE
X_train_pad = encode_texts(tokenizer, X_train.tolist())
X_val_pad   = encode_texts(tokenizer, X_val.tolist())
X_test_pad  = encode_texts(tokenizer, df_test[text_col].tolist())
vocab_size = tokenizer.get_vocab_size() # Get actual size

# --- Label Prep for Focal Loss ---
y_train_cat = utils.to_categorical(y_train, num_classes)
y_val_cat   = utils.to_categorical(y_val, num_classes)
# y_test remains integer for sklearn metrics

# Class weights for training (Phase 2 equivalent in this single-phase run)
class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = {i: w for i, w in enumerate(class_weights_arr)}

# Prepare tf.data.Dataset
def make_dataset(features, labels, batch_size=BATCH_SIZE, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((features, labels))
    if shuffle:
        ds = ds.shuffle(len(features), reshuffle_each_iteration=True)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(X_train_pad, y_train_cat, shuffle=True)
val_ds   = make_dataset(X_val_pad,   y_val_cat)
test_ds  = make_dataset(X_test_pad,  y_test)

# --- Model Instantiation ---
model_b = build_model(vocab_size)
model_b.compile(
    # FIX: Gradient clipping for stability
    optimizer=optimizers.Adam(learning_rate=2e-4, clipnorm=1.0),
    # Use Focal Loss instead of sparse_categorical_crossentropy
    loss=focal_loss(gamma=2.0, alpha=0.75),
    metrics=['accuracy']
)
model_b.summary()

# --- Training / Model Loading Logic ---

if not EVAL_MODE:
    # TRAINING MODE
    print("\n--- Running Training Mode ---")

    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint(MODEL_PATH, monitor='val_loss', save_best_only=True, verbose=1),
        tf.keras.callbacks.TerminateOnNaN() # Added stability callback
    ]

    history = model_b.fit(
        train_ds,
        validation_data=val_ds,
        epochs=30, # Increased epochs to account for complexity
        class_weight=class_weights,
        callbacks=callbacks,
        verbose=1
    )
    print("\n--- Training Complete. Best model saved to:", MODEL_PATH)
    # PLOT TRAINING HISTORY HERE
    plot_training_history(history, filename='training_history_bpe.png')

else:
    # EVALUATION MODE
    print(f"\n--- Running Evaluation Mode: Loading weights from {MODEL_PATH} ---")
    try:
        # Load the entire model, including optimizer state and loss function
        model_b = tf.keras.models.load_model(
            MODEL_PATH,
            custom_objects={"loss": focal_loss()} # Need to pass custom object for Focal Loss
        )
        print("Model loaded successfully.")
    except Exception as e:
        print(f"ERROR: Could not load saved model from {MODEL_PATH}. Error: {e}")
        sys.exit(1)


# Evaluation Block (Runs after training or loading)
print("\n--- Starting Evaluation ---")

# Collect predictions (Model expects padded IDs)
print("Predicting on Validation data...")
val_probs  = model_b.predict(X_val_pad, verbose=0)
print("Predicting on Test data...")
test_probs = model_b.predict(X_test_pad, verbose=0)

val_pred  = val_probs.argmax(axis=1)
test_pred = test_probs.argmax(axis=1)

# Convert one-hot y_val back to integer labels for sklearn
y_val_int = np.argmax(y_val_cat, axis=1)

print("VAL macro-F1:", f1_score(y_val_int, val_pred, average='macro'))
print("TEST macro-F1:", f1_score(y_test, test_pred, average='macro'))

print("\nVAL report:\n", classification_report(y_val_int, val_pred, target_names=EMOTION_LABELS))
print("\nTEST report:\n", classification_report(y_test, test_pred, target_names=EMOTION_LABELS))

# PLOT CONFUSION MATRIX HERE
plot_confusion_matrix(y_test, test_pred, EMOTION_LABELS, filename='test_confusion_matrix_bpe.png')


## Last LSTM with stanza

In [2]:
python -c "import tensorflow as tf; print('TensorFlow:', tf.config.list_physical_devices('GPU')); import stanza; import torch; print('PyTorch (Stanza):', torch.cuda.is_available())"

SyntaxError: invalid syntax (3360177740.py, line 1)

In [ ]:
# ----------------------------
# ENVIRONMENT & LIBRARY SETUP
# ----------------------------
import numpy as np, pandas as pd, random, tensorflow as tf, sys
import re
import os

# Imports for Stanza Tokenization
import stanza
from tensorflow.keras.preprocessing.text import Tokenizer as KerasTokenizer, tokenizer_from_json

# Standard ML/DL Imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from tensorflow.keras import layers, Model, utils, optimizers
import matplotlib.pyplot as plt

# Enable Mixed Precision & XLA (for speed)
tf.keras.mixed_precision.set_global_policy('mixed_float16')
tf.config.optimizer.set_jit(True)

# ----------------------------
# Configuration
# ----------------------------
seed = 42
train_pkl = "FINAL_but_cooler.pkl"
test_csv  = "NLP_features_engineered_but_even_cooler.csv"
text_col  = "text"
label_col = "emotion"
test_text_col  = "Sentence"
test_label_col = "Read emotions"

# --- Model/Data Constants ---
MAX_LEN = 120
EMBEDDING_DIM = 128
VOCAB_SIZE = 35000
BATCH_SIZE = 128
# UPDATED Paths for Stanza
KERAS_TOKENIZER_PATH = 'keras_tokenizer.json'
MODEL_PATH = 'best_stanza_focal.keras'
PERIODIC_CHECKPOINT_PATH = 'checkpoints/epoch_{epoch:02d}.keras'


# Set seeds for reproducibility
np.random.seed(seed); random.seed(seed); tf.random.set_seed(seed)
EVAL_MODE = '--evaluate' in sys.argv

print(f"--- Starting Script (Evaluation Mode: {EVAL_MODE}) ---")

# ----------------------------
# 1. BULGARIAN PREPROCESSING
# ----------------------------

def preprocess_bulgarian_text(text):
    """Removes URLs/noise and keeps Cyrillic/Latin characters."""
    if pd.isna(text) or not isinstance(text, str):
        return ""
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # Keep Cyrillic, Latin, spaces, and essential punctuation
    text = re.sub(r'[^\u0400-\u04FFa-zA-Z\s.,!?;:]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()


# ----------------------------
# 2. BPE TOKENIZATION FUNCTIONS (REMOVED)
#    Stanza + KerasTokenizer logic is now in the main script body.
# ----------------------------


# ----------------------------
# 3. CUSTOM FOCAL LOSS FUNCTION
# ----------------------------
def focal_loss(gamma=2.0, alpha=0.75):
    """Focal Loss for classification, critical for extreme imbalance."""
    def loss(y_true, y_pred):
        # y_true must be one-hot (categorical)
        y_true = tf.cast(y_true, dtype=tf.float32)

        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)

        # Cross-entropy part
        ce = -y_true * tf.math.log(y_pred)
        # Weighting factor: alpha * (1 - p)^gamma
        weight = alpha * tf.pow(1 - y_pred, gamma) * y_true

        return tf.reduce_mean(tf.reduce_sum(weight * ce, axis=1))
    return loss


# ----------------------------
# 4. PLOTTING UTILITY FUNCTIONS
# ----------------------------

# UPDATED: Changed default filename
def plot_confusion_matrix(y_true, y_pred, class_names, filename='test_confusion_matrix_stanza.png'):
    """Calculates, plots, and saves the confusion matrix as a PNG file."""
    cm = confusion_matrix(y_true, y_pred)
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    plt.figure(figsize=(10, 8))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title("Confusion Matrix")
    plt.colorbar()

    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45)
    plt.yticks(tick_marks, class_names)

    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, f"{cm[i, j]}\n({cm_normalized[i, j]:.2f})",
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.savefig(filename)
    plt.close()
    print(f"\n[INFO] Saved Confusion Matrix to {filename}")


# UPDATED: Changed default filename
def plot_training_history(history, filename='training_history_stanza.png'):
    """Plots and saves the loss curves over training epochs (Train vs Val Loss)."""
    train_loss = history.history['loss']
    val_loss = history.history['val_loss']

    plt.figure(figsize=(10, 6))
    plt.plot(train_loss, label='Training Loss', color='blue')
    plt.plot(val_loss, label='Validation/Test Loss', color='orange')
    plt.title('Training vs Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.savefig(filename)
    plt.close()
    print(f"\n[INFO] Saved Training History Plot to {filename}")

# ----------------------------
# 5. MODEL ARCHITECTURE
# ----------------------------
def build_model(vocab_size, num_classes):
    """Defines the stacked Bi-LSTM + Multi-Head Attention architecture."""
    inputs = layers.Input(shape=(MAX_LEN,), dtype='int32')

    # 1. Fully Trainable Embedding Layer
    #    mask_zero=True is important as 0 is the padding index
    x = layers.Embedding(vocab_size, EMBEDDING_DIM, mask_zero=True)(inputs)

    # 2. Stacked Bi-LSTMs (Wider, Increased Dropout)
    x = layers.Bidirectional(layers.LSTM(384, return_sequences=True, dropout=0.3))(x)
    x = layers.LayerNormalization()(x) # LayerNorm stabilizes stacked LSTMs
    x = layers.Bidirectional(layers.LSTM(384, return_sequences=True, dropout=0.3))(x)
    x = layers.LayerNormalization()(x)

    # 3. Multi-Head Attention
    attn = layers.MultiHeadAttention(num_heads=8, key_dim=64)(x, x)
    x = layers.Add()([x, attn])
    x = layers.LayerNormalization()(x)

    # 4. Aggregation (Sentence Embedding)
    max_pool = layers.GlobalMaxPooling1D()(x)
    avg_pool = layers.GlobalAveragePooling1D()(x)
    x = layers.Concatenate()([max_pool, avg_pool])

    # 5. Classifier Head
    x = layers.Dense(512, activation='swish', kernel_initializer='he_normal')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='swish', kernel_initializer='he_normal')(x)
    x = layers.Dropout(0.4)(x)

    # Final output layer (dtype='float32' for stability with mixed precision)
    outputs = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)
    return Model(inputs=inputs, outputs=outputs)


# ----------------------------
# MAIN EXECUTION
# ----------------------------
# Load data
df_train = pd.read_pickle(train_pkl)[[text_col, label_col]].dropna(subset=[text_col, label_col])
df_test  = pd.read_csv(test_csv)[[test_text_col, test_label_col]].dropna(subset=[test_text_col, test_label_col])
df_test  = df_test.rename(columns={test_text_col: text_col, test_label_col: label_col})

# Apply preprocessing to text columns
df_train[text_col] = df_train[text_col].apply(preprocess_bulgarian_text)
df_test[text_col] = df_test[text_col].apply(preprocess_bulgarian_text)

# Filter out empty texts after cleaning
df_train = df_train[df_train[text_col].str.len() > 0]
df_test = df_test[df_test[text_col].str.len() > 0]

print("Train size:", len(df_train), "Test size:", len(df_test))

# --- Stanza Tokenization ---
# This block replaces the BPE training.
# We tokenize all texts first, then use KerasTokenizer to build a vocabulary.
print("Downloading/Loading Stanza Bulgarian model ('bg')...")
stanza.download('bg')
nlp_pipeline = stanza.Pipeline('bg', processors='tokenize', verbose=False)
print("Stanza model loaded.")

def tokenize_with_stanza(nlp, text_series):
    """Helper function to tokenize a pandas Series using Stanza."""
    tokenized_list = []
    for doc in nlp.pipe(text_series.tolist()):
        # Join all tokens in all sentences with a space
        tokenized_list.append(" ".join([token.text for sent in doc.sentences for token in sent.tokens]))
    return tokenized_list

print("Tokenizing training texts with Stanza (this may take a while)...")
df_train['tokenized_text'] = tokenize_with_stanza(nlp_pipeline, df_train[text_col])
print("Tokenizing test texts with Stanza...")
df_test['tokenized_text'] = tokenize_with_stanza(nlp_pipeline, df_test[text_col])
print("Stanza tokenization complete.")


# --- Label Encoding ---
encoder = LabelEncoder()
y_all = encoder.fit_transform(df_train[label_col])
num_classes = len(encoder.classes_)
EMOTION_LABELS = encoder.classes_

# Split data (using the new tokenized text)
X_train, X_val, y_train, y_val = train_test_split(
    df_train['tokenized_text'], y_all, test_size=0.2, random_state=seed, stratify=y_all
)
# Encode test labels
y_test = encoder.transform(df_test[label_col])

# --- Keras Tokenizer Setup ---
if os.path.exists(KERAS_TOKENIZER_PATH):
    print(f"Loading Keras tokenizer from {KERAS_TOKENIZER_PATH}...")
    with open(KERAS_TOKENIZER_PATH, 'r', encoding='utf-8') as f:
        tokenizer_json = f.read()
    tokenizer = tokenizer_from_json(tokenizer_json)
else:
    print("Fitting Keras tokenizer...")
    # Use VOCAB_SIZE as num_words. This keeps the top (VOCAB_SIZE - 1) words
    # and uses 1 for the OOV token.
    tokenizer = KerasTokenizer(num_words=VOCAB_SIZE, oov_token="[UNK]")
    tokenizer.fit_on_texts(X_train)

    # Save the tokenizer
    tokenizer_json = tokenizer.to_json()
    with open(KERAS_TOKENIZER_PATH, 'w', encoding='utf-8') as f:
        f.write(tokenizer_json)
    print(f"Keras tokenizer saved to {KERAS_TOKENIZER_PATH}.")

# Encode and Pad sequences using Keras Tokenizer
print("Encoding and padding sequences...")
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq   = tokenizer.texts_to_sequences(X_val)
X_test_seq  = tokenizer.texts_to_sequences(df_test['tokenized_text'].tolist())

# Keras pad_sequences defaults to value=0, which is correct.
X_train_pad = tf.keras.preprocessing.sequence.pad_sequences(
    X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post'
)
X_val_pad = tf.keras.preprocessing.sequence.pad_sequences(
    X_val_seq, maxlen=MAX_LEN, padding='post', truncating='post'
)
X_test_pad = tf.keras.preprocessing.sequence.pad_sequences(
    X_test_seq, maxlen=MAX_LEN, padding='post', truncating='post'
)

# The vocab size for the Embedding layer is the number of words specified
vocab_size = VOCAB_SIZE

# --- Label Prep for Focal Loss ---
y_train_cat = utils.to_categorical(y_train, num_classes)
y_val_cat   = utils.to_categorical(y_val, num_classes)
# y_test remains integer for sklearn metrics

# Class weights
class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = {i: w for i, w in enumerate(class_weights_arr)}

# Prepare tf.data.Dataset
def make_dataset(features, labels, batch_size=BATCH_SIZE, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((features, labels))
    if shuffle:
        ds = ds.shuffle(len(features), reshuffle_each_iteration=True)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(X_train_pad, y_train_cat, shuffle=True)
val_ds   = make_dataset(X_val_pad,   y_val_cat)
test_ds  = make_dataset(X_test_pad,  y_test)

# --- Model Instantiation ---
model_b = build_model(vocab_size, num_classes)
model_b.compile(
    optimizer=optimizers.Adam(learning_rate=2e-4, clipnorm=1.0),
    loss=focal_loss(gamma=2.0, alpha=0.75),
    metrics=['accuracy']
)
model_b.summary()

# --- Training / Model Loading Logic ---

if not EVAL_MODE:
    # TRAINING MODE
    print("\n--- Running Training Mode ---")

    # Create directory for periodic checkpoints
    os.makedirs('checkpoints', exist_ok=True)

    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
        # Callback to save the BEST model
        tf.keras.callbacks.ModelCheckpoint(MODEL_PATH, monitor='val_loss', save_best_only=True, verbose=1),
        # NEW Callback to save a checkpoint every 2 epochs
        tf.keras.callbacks.ModelCheckpoint(PERIODIC_CHECKPOINT_PATH, period=2, save_best_only=False, verbose=1),
        tf.keras.callbacks.TerminateOnNaN()
    ]

    history = model_b.fit(
        train_ds,
        validation_data=val_ds,
        epochs=30,
        class_weight=class_weights,
        callbacks=callbacks,
        verbose=1
    )
    print("\n--- Training Complete. Best model saved to:", MODEL_PATH)
    plot_training_history(history, filename='training_history_stanza.png')

else:
    # EVALUATION MODE
    print(f"\n--- Running Evaluation Mode: Loading weights from {MODEL_PATH} ---")
    try:
        model_b = tf.keras.models.load_model(
            MODEL_PATH,
            custom_objects={"loss": focal_loss()}
        )
        print("Model loaded successfully.")
    except Exception as e:
        print(f"ERROR: Could not load saved model from {MODEL_PATH}. Error: {e}")
        sys.exit(1)


# Evaluation Block (Runs after training or loading)
print("\n--- Starting Evaluation ---")

print("Predicting on Validation data...")
val_probs  = model_b.predict(X_val_pad, verbose=0)
print("Predicting on Test data...")
test_probs = model_b.predict(X_test_pad, verbose=0)

val_pred  = val_probs.argmax(axis=1)
test_pred = test_probs.argmax(axis=1)

print("VAL macro-F1:", f1_score(y_val, val_pred, average='macro'))
print("TEST macro-F1:", f1_score(y_test, test_pred, average='macro'))

print("\nVAL report:\n", classification_report(y_val, val_pred, target_names=EMOTION_LABELS))
print("\nTEST report:\n", classification_report(y_test, test_pred, target_names=EMOTION_LABELS))

# PLOT CONFUSION MATRIX
plot_confusion_matrix(y_test, test_pred, EMOTION_LABELS, filename='test_confusion_matrix_stanza.png')